# Q1 — o tamanho da diferença

Objetivo: investigar a diferença nas notas da prova de Ciências da Natureza (CN) do ENEM entre as redes de ensino, as faixas de renda per capita e os níveis de escolaridade dos pais. Essa diferença muda ao longo do tempo?

- **Por renda e escolaridade dos pais:** só de 2020 a 2023 (`fct_individual`), porque em 2024 e 2025 a nota não se liga mais ao questionário socioeconômico, como explicado na D002.

- **Por rede:** de 2020 a 2025 (`agg_rede_ano`), com atenção à quebra de série entre 2023 e 2024, também explicada na D002.

## Configurações iniciais

In [ ]:
from src.db import conectar
con = conectar(read_only=True)
from src import graficos as g
g.aplicar_tema()


## Análise entre as diferentes redes de ensino

Começando pela análise entre as redes de ensino, vamos utilizar a *tabela* `agg_rede_ano`, selecionando as colunas com o `ano`, o período (`periodo`), a `rede`, a quantidade de participantes com nota em Ciências da Natureza (`qtd_com_nota_cn`), a média da nota em Ciências da Natureza (`media_nota_cn`), a mediana (`p50_nota_cn`) e o desvio-padrão da nota em Ciências da Natureza (`desvio_padrao_nota_cn`), ordenadas pela `rede` e pelo `ano`.

In [ ]:
rede = con.sql("""
    SELECT ano, periodo, rede,
        qtd_com_nota_cn, media_nota_cn, p50_nota_cn,
        desvio_padrao_nota_cn
    FROM agg_rede_ano
    ORDER BY rede, ano
""").df()
rede

### A quebra de série entre 2023 e 2024

Como explicado na D002, entre 2023 e 2024 o INEP alterou a forma como as informações sobre a escola do participante são obtidas. Até 2023, era responsabilidade do participante preencher essas informações; a partir de 2024, elas passaram a ser obtidas do Censo Escolar, a partir do CPF do participante.

Como efeito imediato dessa mudança, há uma quebra de série visível nos dados, principalmente nas escolas **municipais** e **estaduais** (`rede=estadual_municipal`). Essa mudança pode ser vista com clareza na coluna `qtd_com_nota_cn`:

| Rede | 2023 | 2024 | Variação |
|------------|----------|---------|---------|
| Estadual/municipal | 28.688 | 73.200 | +155% (2,5 vezes) |
| Federal | 5.660 | 7.370 | +30% |
| Privada | 18.378 | 21.430 | +17% |

A rede estadual/municipal mais que dobrou, enquanto as outras cresceram entre 17% e 30%.

### A nota média por rede de ensino

Vamos montar um gráfico para analisar a nota média de Ciências da Natureza ao longo do tempo, levando em consideração a quebra de série descrita anteriormente:

In [ ]:
fig, ax = g.serie_temporal(
    rede, x="ano", y="media_nota_cn", grupo="rede",
    rotulos={"federal": "Federal",
            "estadual_municipal": "Estadual/municipal",
            "privada": "Privada"},
    marcar_quebra=True,
    rotulo_quebra="Mudança na fonte da escola",
    conectar_quebra=False,
    legenda=False,
    titulo="Média em Ciências da Natureza por rede de ensino — MG, 2020–2025",
    subtitulo="Concluintes de escolas mineiras com nota na prova",
    rotulo_y="Nota média (pontos)",
    nota="Fonte: microdados do ENEM (INEP). Até 2023, a escola era informada "
        "pelo participante; a partir de 2024, vem do Censo Escolar.",
    figsize=(8, 5),
)
g.salvar(fig, "q1_media_cn_por_rede")

In [ ]:
var_anos = (rede[rede["periodo"] == "2020-2023"]
            .groupby("rede")["media_nota_cn"]
            .agg(desvio_padrao="std",
                minimo="min",
                maximo="max")
            .assign(amplitude=lambda d: d["maximo"] - d["minimo"])
            .round(1))
print(var_anos.to_string())

Entre 2020 e 2023, a variação das notas pode ser mais bem observada na tabela:

| Rede | Faixa em 2020–2023 | Amplitude | Desvio-padrão da média |
|-------|------------|-----------|-----------|
| Federal | de 551,6 (2023) a 569,4 (2020) | 17,8 pontos | 8,0 |
| Privada | de 553,6 (2022) a 570,3 (2020) | 16,7 pontos | 8,2 |
| Estadual/municipal | de 475,1 (2021) a 491,6 (2023) | 16,5 pontos | 7,0 |

As três redes variam de forma parecida no período observado.

A quebra de série entre 2023 e 2024 pode explicar a queda da nota média da rede estadual/municipal entre esses anos. Como a quantidade de participantes mais que dobrou de um ano para o outro, a queda é compatível com um efeito de composição da amostra, e não com uma piora real do desempenho dos estudantes. Já nas redes federal e privada, nas quais o número de participantes cresceu relativamente menos (entre 17% e 30%), esse efeito é bem menos sentido.

Vamos comparar agora as notas das três redes de ensino, ano a ano:

In [ ]:
gaps = (rede.pivot(index="ano", columns="rede", values="media_nota_cn")
        .assign(privada_menos_estadual=lambda d: d["privada"] - d["estadual_municipal"],
                federal_menos_estadual=lambda d: d["federal"] - d["estadual_municipal"],
                privada_menos_federal=lambda d: d["privada"] - d["federal"])
        .round(1))
print(gaps.to_string())

O gap entre a rede federal e a rede privada é quase zero entre 2020 e 2022 (as duas linhas praticamente se sobrepõem no gráfico), mas a separação entre elas fica mais evidente em 2023, quando a média da rede federal parece continuar diminuindo (veja o desenvolvimento dessa hipótese mais à frente no notebook) e a da rede privada aumenta. Nesse ano, o gap entre as duas passa a ser de 18,2 pontos.

As médias da rede estadual/municipal crescem de 2021 a 2023, o que pode indicar uma melhora no desempenho da rede, mas são necessárias mais análises para avaliar essa hipótese.

Em 2020, as médias das três redes são maiores que as do ano seguinte. O ano de 2020 teve abstenção recorde (55%), por causa da pandemia de covid-19. As médias de 2020 podem ter sido infladas se houve algum viés de seleção entre os participantes que compareceram à prova.

### Distribuição das notas por rede de ensino

In [ ]:
rotulos_rede = {"federal": "Federal",
                "estadual_municipal": "Estadual/municipal",
                "privada": "Privada"}

notas = con.sql("""
    SELECT ano, rede, nota_cn
    FROM fct_agregada
    WHERE tem_nota_cn AND ano IN (2023, 2025)
""").df()

for ano in (2023, 2025):
    fig, ax = g.distribuicoes(
        notas[notas["ano"] == ano],
        coluna_valor="nota_cn",
        coluna_grupo="rede",
        rotulos=rotulos_rede,
        mostrar_n=True,
        intervalo=(290, 850),
        preencher=False,
        titulo=f"Distribuição das notas em Ciências da Natureza por rede — MG, {ano}",
        subtitulo="Concluintes de escolas mineiras com nota na prova",
        rotulo_x="Nota em Ciências da Natureza (pontos)",
        rotulo_y="Densidade",
        nota="Fonte: microdados do ENEM (INEP). Curvas normalizadas: cada rede tem área 1.",
    )
    g.salvar(fig, f"q1_distribuicao_cn_por_rede_{ano}")



A curva da rede estadual/municipal está deslocada para a esquerda em relação às curvas das outras redes, que são praticamente sobrepostas em 2023 e em 2025. Tomando a rede privada como referência nos dois anos, o deslocamento da mediana de cada uma das outras redes em relação à mediana da privada pode ser visto na tabela:

| Rede | Ano | Deslocamento da mediana (pontos) |
|----|-----|--------------|
| Estadual/municipal | 2023 | −80,3 |
| Estadual/municipal | 2025 | −95,4 |
| Federal | 2023 | −15,8 |
| Federal | 2025 | −11,0 |

Em 2023 e em 2025, a média é menor que a mediana das notas de Ciências da Natureza em todas as redes, como mostra a tabela abaixo.

In [ ]:
resumo = (notas.groupby(["ano", "rede"])["nota_cn"]
            .agg(media="mean", mediana="median")
            .assign(media_menos_mediana=lambda d: d["media"] - d["mediana"])
            .round(1))
print(resumo.to_string())

Na rede estadual/municipal, a diferença tem módulo menor que 1, o que indica curvas mais simétricas. Vale destacar também que a distribuição de 2023 apresenta um pequeno pico perto de 340 pontos, que não se repete em 2025. Nas redes privada e federal, a diferença entre a média e a mediana é relativamente maior, mas ainda assim pequena. As diferenças vão de 0,8 a 5,3 pontos, enquanto o desvio-padrão das notas fica em torno de 66 a 79 pontos. Ou seja, as diferenças entre média e mediana equivalem a, no máximo, cerca de 7% de um desvio-padrão. A tabela abaixo mostra isso:

In [ ]:
resumo_dp = (notas.groupby(["ano", "rede"])["nota_cn"]
                .agg(media="mean", mediana="median", dp="std")
                .assign(dif_em_dp=lambda d: (d["media"] - d["mediana"]) / d["dp"])
                .round(3))
print(resumo_dp.to_string())

Na tabela, a coluna `dif_em_dp` coloca a diferença na escala do desvio-padrão, o que torna as redes e os anos comparáveis. As caudas de notas mais baixas existem em todos os casos, mas são sutis.

### A estabilidade da rede federal

Como mostra a figura `../reports/figures/q1_media_cn_por_rede.png`, as médias em Ciências da Natureza das redes federal e privada seguem muito próximas de 2020 a 2022. Em 2023, a média da rede privada cresce, e a da rede federal parece continuar diminuindo. Nesse ano, o gap entre as duas é de 18,2 pontos.

A rede federal é a que tem menos participantes no ENEM entre as três redes. As variações de um ano para o outro são mudanças reais ou flutuações de uma amostra pequena? Vamos observar o erro-padrão da média:

In [ ]:
ep = (rede.assign(erro_padrao=lambda d: d["desvio_padrao_nota_cn"] / d["qtd_com_nota_cn"]**0.5)
        [["ano", "rede", "qtd_com_nota_cn", "media_nota_cn", "erro_padrao"]]
        .round(2))
print(ep[ep["rede"] == "federal"].to_string())

O erro-padrão da federal fica em torno de 1 ponto em todos os anos. Vamos compará-lo com as variações entre anos seguidos (exceto de 2023 para 2024, por causa da quebra de série):

In [ ]:
fed = ep.merge(rede[["ano", "rede", "periodo"]], on=["ano", "rede"])
fed = fed[fed["rede"] == "federal"].sort_values("ano").reset_index(drop=True)
mesmo_periodo = fed["periodo"] == fed["periodo"].shift()
fed = fed.assign(
    variacao=fed["media_nota_cn"].diff().where(mesmo_periodo),
    ep_diferenca=((fed["erro_padrao"]**2 + fed["erro_padrao"].shift()**2)**0.5).where(mesmo_periodo),
)
fed = fed.assign(razao=fed["variacao"].abs() / fed["ep_diferenca"]).round(2)
print(fed[["ano", "periodo", "media_nota_cn", "variacao", "ep_diferenca", "razao"]].to_string())

A coluna `razao` diz quantos erros-padrão a variação representa. Acima de 2, a variação dificilmente se deve ao acaso. Pela tabela, a queda de 2022 para 2023 (−1,98 ponto) é do tamanho da flutuação esperada. Não podemos afirmar que a nota da rede federal continuou caindo, e sim que ela ficou estável. O gap entre a federal e a privada em 2023 deve-se, então, principalmente à subida da rede privada, e não a uma queda da rede federal.

As grandes mudanças nas notas da rede federal (de 2020 para 2021 e de 2024 para 2025) são reais, no sentido de não serem acaso. As menores (de 2021 para 2022 e de 2022 para 2023) estão no limite ou dentro da flutuação esperada. O motivo das mudanças reais continua em aberto. Além da hipótese da pandemia, levantada anteriormente para 2020, o perfil de quem fez a prova e até mesmo diferenças entre as provas podem contribuir para essas mudanças.

### A inflação das médias de 2020

A abstenção recorde de 2020 (55%) pode ter inflado as médias desse ano? Se os alunos que compareceram à prova eram, de alguma forma, mais preparados do que os que se abstiveram, as médias de 2020 podem ter ficado artificialmente altas.

Para testar essa hipótese, vamos comparar a proporção de inscritos que chegaram a ter nota, em cada rede, em 2020 e em 2021:

In [ ]:
con.sql("""
    SELECT ano, rede,
        qtd_inscritos, qtd_com_nota_cn,
           round(100.0 * qtd_com_nota_cn / qtd_inscritos, 1) AS pct_com_nota
    FROM agg_rede_ano
    WHERE ano IN (2020, 2021)
    ORDER BY rede, ano
""").show()

| Rede | % com nota (2020 → 2021) | Diferença | Média (2020 → 2021) | Diferença |
|---|---|---|---|---|
| Estadual/municipal | 55,2% → 69,1% | +13,9 p.p. | 486,4 → 475,1 | −11,3 |
| Privada | 85,8% → 95,2% | +9,4 p.p. | 570,3 → 559,2 | −11,1 |
| Federal | 87,0% → 92,3% | +5,3 p.p. | 569,4 → 556,9 | −12,4 |

Em 2020, ficou mais gente fora da média nas três redes, sobretudo na estadual/municipal, onde só 55% dos inscritos chegaram a ter nota. Se os que faltaram tinham, em geral, desempenho esperado menor, as médias de 2020 ficaram infladas. No entanto, a tabela também mostra que, na rede com a **maior** mudança na presença (estadual/municipal, +13,9 p.p.), a média cai 11,3 pontos, enquanto na rede com a **menor** mudança (federal, +5,3 p.p.) a média cai ainda mais, cerca de 12 pontos.

A queda das médias é parecida nas três redes, mesmo com mudanças muito diferentes na presença. Isso sugere que boa parte da queda de 2020 para 2021 vem de algo comum a todas as redes, como uma diferença entre as provas dos dois anos ou um efeito geral da pandemia sobre todos os alunos, e não só da seleção de quem compareceu.

Além disso, aparece aqui uma diferença interessante entre as redes. Em 2021, tomado aqui como um ano mais "normal", 69% dos inscritos da rede estadual/municipal tiveram nota, contra 92% na federal e 95% na privada. Esse gap de presença é, por si só, uma forma de desigualdade, que pode ser investigada mais adiante na Q4 (veja `../docs/perguntas.md`).

### Conclusões da análise por rede de ensino

#### O que os dados mostram
- A rede estadual/municipal fica bem abaixo das outras duas em todos os anos. A diferença para a privada vai de 67 a 93 pontos, e para a federal, de 60 a 83 pontos, o que equivale a cerca de um desvio padrão das notas. Nas distribuições, a mediana da estadual/municipal fica 80 pontos abaixo da mediana da privada em 2023 e 95 pontos abaixo em 2025.
- Federal e privada andam juntas até 2022 e se separam em 2023. O gap passa de praticamente zero para 18,2 pontos. A separação vem da subida da privada: a variação da federal de 2022 para 2023 está dentro da flutuação esperada.
- As distribuições são quase simétricas nas três redes. A média fica um pouco abaixo da mediana em todos os casos, mas a diferença não passa de 7% do desvio padrão: as caudas de notas baixas existem, mas são sutis.
- A presença na prova também é desigual. Em 2021, 69% dos inscritos da estadual/municipal tiveram nota, contra 92% na federal e 95% na privada.

#### O que exige cuidado na leitura
- A quebra de série entre 2023 e 2024 impede a comparação entre os dois períodos. A queda da estadual/municipal de 2023 para 2024 coincide com um aumento de 155% no número de participantes com nota, o que é compatível com um efeito de composição, e não necessariamente com uma piora real.
- 2020 é um ano atípico. A absetenção recorde é compatível com as médias infladas pela seleção de quem compareceu, mas a queda de 2020 para 2021 é parecida nas três redes (de 11 a 12 pontos), mesmo com mudanças muito diferentes na presença. Isso sugere um fator comum a todas as redes, além da seleção.
- Variações reais não indicam, sozinhas, mudança na qualidade das redes. As grandes oscilações da federal (de 2020 para 2021 e de 2024 para 2025) não são acaso, mas podem vir do perfil de quem fez a prova ou de diferenças entre as provas de cada ano.
- As diferenças entre redes não são efeito da rede em si. As redes atendem alunos com origens diferentes, e parte da rede federal seleciona alunos por prova. Separar o efeito da rede do efeito da origem é o objetivo da Q3.


#### O que fica em aberto
- Por que a rede privada sobe em 2023.
- Se a subida da estadual/municipal de 2021 a 2023 representa uma melhora real.
- A desigualdade de presença entre as redes, que segue para a Q4.
- A diferença por renda e por escolaridade dos pais, próximas partes desta Q1.